# 10.数据聚合与分类操作

# 10.1 GroupBy机制

In [11]:
import numpy as np
import pandas as pd
df = pd.DataFrame({"key1" : ["a", "a", None, "b", "b", "a", None],
                   "key2" : pd.Series([1, 2, 1, 2, 1, None, 1],
                                      dtype="Int64"),
                   "data1" : np.random.standard_normal(7),
                   "data2" : np.random.standard_normal(7)})
df

,key1,key2,data1,data2
0,a,1,-1.154192,-0.170960
1,a,2,0.257856,-2.002642
2,NaN,1,0.342443,2.653869
3,b,2,-0.158658,0.746597
4,b,1,-1.086919,0.683743
5,a,<NA>,-1.071087,-1.066523
6,NaN,1,0.057349,-0.195875


In [12]:
grouped = df['data1'].groupby(df['key1'])
grouped

In [13]:
grouped.mean()

key1
a   -0.655807
b   -0.622788
Name: data1, dtype: float64

In [14]:
means = df['data1'].groupby([df['key1'],df['key2']]).mean()
means

key1  key2
a     1      -1.154192
      2       0.257856
b     1      -1.086919
      2      -0.158658
Name: data1, dtype: float64

In [15]:
means.unstack()

key2,1,2
key1,,
a,-1.154192,0.257856
b,-1.086919,-0.158658


In [16]:
states = np.array(['OH', 'CA', 'CA', 'OH', 'OH', 'CA', 'OH'])
years = [2005, 2005, 2006, 2005, 2006, 2005, 2006]
df['data1'].groupby([states, years]).mean()

CA  2005   -0.406615
    2006    0.342443
OH  2005   -0.656425
    2006   -0.514785
Name: data1, dtype: float64

In [17]:
# 只传列名也可以
df.groupby('key1').mean()

,key2,data1,data2
key1,,,
a,1.5,-0.655807,-1.080042
b,1.5,-0.622788,0.715170


In [19]:
df.groupby('key2').mean(numeric_only=True)

,data1,data2
key2,,
1,-0.460330,0.742694
2,0.049599,-0.628022


In [20]:
df.groupby(['key1', 'key2']).mean()

data1     data2
key1 key2                    
a    1    -1.154192 -0.170960
     2     0.257856 -2.002642
b    1    -1.086919  0.683743
     2    -0.158658  0.746597

In [21]:
df.groupby(['key1', 'key2']).size()

key1  key2
a     1       1
      2       1
b     1       1
      2       1
dtype: int64

In [22]:
# 传入dropna=False选项可以保留NA值 默认除去任何分组的默认值
df.groupby('key1', dropna=False).size()

key1
a      3
b      2
NaN    2
dtype: int64

In [23]:
df.groupby(['key1', 'key2'], dropna=False).size()

key1  key2
a     1       1
      2       1
      <NA>    1
b     1       1
      2       1
NaN   1       2
dtype: int64

In [24]:
df.groupby('key1').count()

,key2,data1,data2
key1,,,
a,2,3,3
b,2,2,2


## 10.1.1 对分组进行迭代

In [26]:
df

,key1,key2,data1,data2
0,a,1,-1.154192,-0.170960
1,a,2,0.257856,-2.002642
2,NaN,1,0.342443,2.653869
3,b,2,-0.158658,0.746597
4,b,1,-1.086919,0.683743
5,a,<NA>,-1.071087,-1.066523
6,NaN,1,0.057349,-0.195875


In [25]:
for name, group in df.groupby('key1'):
    print(name)
    print(group)

a
  key1  key2     data1     data2
0    a     1 -1.154192 -0.170960
1    a     2  0.257856 -2.002642
5    a  <NA> -1.071087 -1.066523
b
  key1  key2     data1     data2
3    b     2 -0.158658  0.746597
4    b     1 -1.086919  0.683743


In [27]:
for (k1,k2), group in df.groupby(['key1', 'key2']):
    print((k1, k2))
    print(group)

('a', 1)
  key1  key2     data1    data2
0    a     1 -1.154192 -0.17096
('a', 2)
  key1  key2     data1     data2
1    a     2  0.257856 -2.002642
('b', 1)
  key1  key2     data1     data2
4    b     1 -1.086919  0.683743
('b', 2)
  key1  key2     data1     data2
3    b     2 -0.158658  0.746597


In [29]:
pieces = {name: group for name, group in df.groupby('key1')}
pieces['b']

,key1,key2,data1,data2
3,b,2,-0.158658,0.746597
4,b,1,-1.086919,0.683743


## 10.1.2 选取一列或多列

In [33]:
# df.groupby('key1')['data1']  是 df['data1'].groupby(df['key1'])的语法糖 返回SeiresGroupBy对象
# df.groupby('key1')[['data1']] 是 df[['data1']].groupby(df['key1'])的语法糖 返回DataFrameGroupBy对象
df.groupby(['key1', 'key2'])[['data2']].mean()

data2
key1 key2          
a    1    -0.170960
     2    -2.002642
b    1     0.683743
     2     0.746597

In [34]:
df.groupby(['key1', 'key2'])['data2'].mean()

key1  key2
a     1      -0.170960
      2      -2.002642
b     1       0.683743
      2       0.746597
Name: data2, dtype: float64

## 10.1.3 利用字典和Series进行分组

In [35]:
people = pd.DataFrame(np.random.standard_normal((5, 5)),
                      columns=["a", "b", "c", "d", "e"],
                      index=["Joe", "Steve", "Wanda", "Jill", "Trey"])
people.iloc[2:3, [1, 2]] = np.nan # Add a few NA values
people

,a,b,c,d,e
Joe,-1.567004,1.828062,-0.864051,-0.388199,-0.450476
Steve,0.772265,-0.466570,0.354516,0.088930,-0.061373
Wanda,-1.873707,NaN,NaN,-0.136333,-0.975480
Jill,2.155872,0.404242,-0.377649,-1.387440,0.128693
Trey,1.917402,0.794184,0.258056,-1.743674,-0.677574


In [40]:
mapping = {"a": "red", "b": "red", "c": "blue",
           "d": "blue", "e": "red", "f": "orange"}  # 即使存在未分组的键也是可以的
by_column = people.T.groupby(mapping)
by_column.sum()

,Joe,Steve,Wanda,Jill,Trey
blue,-1.252250,0.443446,-0.136333,-1.765090,-1.485618
red,-0.189419,0.244322,-2.849187,2.688807,2.034013


In [41]:
map_series = pd.Series(mapping)
map_series

a       red
b       red
c      blue
d      blue
e       red
f    orange
dtype: str

In [43]:
people.T.groupby(map_series).count().T

,blue,red
Joe,2,3
Steve,2,3
Wanda,1,2
Jill,2,3
Trey,2,3


## 10.1.4 利用函数进行分组

In [44]:
# 函数会在索引上全部执行一次
people.groupby(len).sum()

,a,b,c,d,e
3,-1.567004,1.828062,-0.864051,-0.388199,-0.450476
4,4.073275,1.198427,-0.119594,-3.131114,-0.548881
5,-1.101442,-0.466570,0.354516,-0.047404,-1.036853


In [45]:
# 函数、数组、字典、Series都可以混合使用
key_list = ["one", "one", "one", "two", "two"]
people.groupby([len, key_list]).min()

,,a,b,c,d,e
3,one,-1.567004,1.828062,-0.864051,-0.388199,-0.450476
4,two,1.917402,0.404242,-0.377649,-1.743674,-0.677574
5,one,-1.873707,-0.466570,0.354516,-0.136333,-0.975480


## 10.1.5 根据索引层级分组

In [46]:
columns = pd.MultiIndex.from_arrays([["US", "US", "US", "JP", "JP"],
                                [1, 2, 3, 1, 2]],
                                names=["cty", "tenor"])
hier_df = pd.DataFrame(np.random.standard_normal((4, 5)), columns=columns)
hier_df

cty          US                            JP          
tenor         1         2         3         1         2
0      0.701902 -0.584624  1.055070 -0.094727 -0.299310
1      1.145661  1.014985  0.069496 -0.180369  1.573100
2     -0.399668  1.782620  0.980489 -1.242135  0.375592
3      0.616111  1.824039  1.039816  0.116009 -0.868740

In [48]:
hier_df.T.groupby(level="cty").count()

,0,1,2,3
cty,,,,
JP,2,2,2,2
US,3,3,3,3


# End